# 0. Initialiseer - refresh external data en import libraries

### 0.1 De df_dim_sensor tabel bevat de volgende informatie
- **locatie**            = de naam van de locatie waar de sensor is geplaats
- **device_name**        = de naam van de sensor die op de label van de fysieke sensor staat
- **datum_geplaatst**      = de datum waarop de sensor in de grond is geplaatst
- **datum_weggehaald**  = de datum waarop de sensor uit de grond is gehaald. Als deze datum na vandaag ligt, dan zit de sensor daar nog in de grond.
- **diepte_plaatsing**     = de diepte waarop de pinnen van de sensor in de grond zijn geplaats (15cm of 30cm diep)
- **leeftijd_plant**     = de leeftijd van de beplanting waar de de sensor is geplaats (0-1, 1-2 of 2-3 jaar)
- **soort_plant**        = de omschrijving van de beplanting waar de sensor is geplaatst
- **locatie_regio**      = de regio van de stad waar de sensor is geplaatst
- **extra_omschrijving** = een extra omschrijving van de locatie waar de sensor is geplaatst (bijv. in de berm, vlakbij bestrating, ..)
- **old**                = voor start van nieuwe project zijn er al sensoren geplaatst, deze hebben de indicatie 'old'. Van deze data weten we niet hoe betrouwbaar deze is.
- **device_name_org**    = voor het koppelen van de device_id's met de device_name's is de originele (= org) benaming aangepast om de koppeling te vereenvoudigen, daarom voor de zekerheid deze kolom wel behouden.
- **device_id**          = de id van de device zoals deze geregistreerd staat in de quantified portal. Middels de device id's kan de data via de quantified API opgehaald worden.

### 0.2 De df_fact_sensor tabel bevat de volgende informatie
- **gateway_receive_time**  = tijdstip waarop sensor permittivity waarde is ontvangen
- **device**                = device_id waar de permittivity waarde vandaan komt
- **value**                 = de gemeten permittivity waarde

### 0.3 De df_KNMI bevat de volgende informatie
- **YYYYMMDD**  = Datum (YYYY=jaar MM=maand DD=dag)
- **FG**        = Etmaalgemiddelde windsnelheid (in 0.1 m/s)
- (**TG**        = Etmaalgemiddelde temperatuur (in 0.1 graden Celsius)) --> <u>vervangen door Zusterhof</u>
- **SQ**        = Zonneschijnduur (in 0.1 uur) berekend uit de globale straling (-1 voor <0.05 uur)
- **DR**        = Duur van de neerslag (in 0.1 uur)
- (**RH**        = Etmaalsom van de neerslag (in 0.1 mm) (-1 voor <0.05 mm)) --> <u>vervangen door Zusterhof</u>
- **PG**        = Etmaalgemiddelde luchtdruk herleid tot zeeniveau (in 0.1 hPa) berekend uit 24 uurwaarden
- **NG**        = Etmaalgemiddelde bewolking (bedekkingsgraad van de bovenlucht in achtsten, 9=bovenlucht onzichtbaar)
- **UG**        = Etmaalgemiddelde relatieve vochtigheid (in procenten)
- **EV24**      = Referentiegewasverdamping (Makkink) (in 0.1 mm)

### 0.4 De df_weerstation_leiden bevat de volgende informatie
- **datum**         = Datum meetwaarde
- **temperatuur**   = Etmaalgemiddelde temperatuur (in 0.1 graden Celsius)
- **neerslag**      = Etmaalsom van de neerslag (in mm) (-1 voor <0.05 mm)

### Notes
- De temperatuur en neerslag wordt van weerstation **Zusterhof** in Leiden gebruikt. De overige weerdata vanuit het KNMI van weerstation **Schiphol**.
- De KNMI data is ook op uurniveau beschikbaar, maar deze is niet geschikt voor de analyse vanwege datakwaliteit, daarom gebruiken we KNMI data op dagniveau. 


In [4]:
# TODO:
# Stappenplan van acties voordat dit notebook gedraaid kan worden

In [8]:
# Import neccesary libraries
from logic_components.data_transformations import resample
import numpy as np
import plotly.express as px
import pandas as pd
from pandas.tseries.offsets import DateOffset

### 0. Refresh & Laad data:
- 1.1 Sensorinformatie vanuit google spreadsheet
- 1.2 Quantified data (= df_fact_sensor)
- 1.3 KNMI data (= df_KNMI)
- 1.4 Weerstation Leiden data (= df_weerstation_leiden)

In [9]:
# 0. Refresh & Laad data
# from api_components import refresh_quantified
from api_components import refresh_external_data
from api_components.data_loading import get_all_data

df_dim_sensor, df_fact_sensor, df_KNMI, df_weerstation_leiden = get_all_data()

### 1. Inspecteer data
- 1.1 Check data info, head and tail
- 1.2 Check data visuals

In [10]:
# 1.1 Check data info, head and tail

# df_dim_sensor
display(df_dim_sensor.info(5))
display(df_dim_sensor.head(5))
display(df_dim_sensor.tail(5))

# df_fact_sensor
display(df_fact_sensor.info(5))
display(df_fact_sensor.head(5))
display(df_fact_sensor.tail(5))

# df_KNMI
display(df_KNMI.info(5))
display(df_KNMI.head(5))
display(df_KNMI.tail(5))

# df_weerstation_leiden
display(df_weerstation_leiden.info(5))
display(df_weerstation_leiden.head(5))
display(df_weerstation_leiden.tail(5))

# 1.2 Check data visuals
df_visual = df_fact_sensor.copy()
df_visual.sort_values(by=['device', 'gateway_receive_time'], inplace=True)

fig = px.line(df_visual, x='gateway_receive_time', y='value', color='device', title='Time Series of all devices',
              labels={'value': 'Permittivity', 'gateway_receive_time': 'Time'},
              line_group='device', hover_name='device')

fig.show()

cutt_off = df_visual['gateway_receive_time'].max() - DateOffset(months=2)
fig = px.line(df_KNMI[df_KNMI['datum'] > cutt_off], x='datum', y='neerslag', title='Time Series of neerslag KNMI')

fig.show()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   locatie            44 non-null     object        
 1   device_name        44 non-null     object        
 2   device_id          44 non-null     int64         
 3   lat                0 non-null      float64       
 4   lon                0 non-null      float64       
 5   coordinaat         44 non-null     object        
 6   datum_geplaatst    44 non-null     datetime64[ns]
 7   datum_weggehaald   44 non-null     object        
 8   soort_plant        44 non-null     object        
 9   leeftijd_plant     44 non-null     object        
 10  diepte_plaatsing   44 non-null     object        
 11  omgevingsfactoren  44 non-null     object        
 12  locatie_regio      44 non-null     object        
 13  old                44 non-null     object        
dtypes: datetime6

None

,locatie,device_name,device_id,lat,lon,coordinaat,datum_geplaatst,datum_weggehaald,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old
0,Burggravenlaan (old)_15,FF1 0-0004,93,NaN,NaN,"(, )",2022-01-28,2023-05-24 00:00:00,Plantenvak,> 1 jaar,15 cm,groen,Binnenstad,Yes
1,Vijf meilaan (2)_30,FF1 0-0004,93,NaN,NaN,"(, )",2023-07-04,2262-04-11 00:00:00,Plantenvak,0-1 jaar,30 cm,bestrating,Zuid,Yes
2,Beethovenpark_30,FF1 0-0027,54,NaN,NaN,"(, )",2023-07-04,2023-08-07 00:00:00,Plantenvak,0-1 jaar,30 cm,groen,Zuid,Yes
3,Beethovenpark_15,FF1 0-0027,54,NaN,NaN,"(, )",2023-08-07,2262-04-11 00:00:00,Plantenvak,0-1 jaar,15 cm,groen,Zuid,Yes
4,Ijselmeerlaan berm (old)_15,FF1 0-0027,54,NaN,NaN,"(, )",2022-01-28,2023-05-01 00:00:00,Heesters,> 1 jaar,15 cm,bestrating,Noord,Yes


,locatie,device_name,device_id,lat,lon,coordinaat,datum_geplaatst,datum_weggehaald,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old
39,Langegracht Politie_15,FF2 0-0191,416,NaN,NaN,"(, )",2023-08-02,2262-04-11 00:00:00,Plant en boom,> 1 jaar,15 cm,bestrating,Binnenstad,No
40,Arendshorst_30,FF2 0-0193,418,NaN,NaN,"(, )",2023-08-02,2262-04-11 00:00:00,Boom,> 1 jaar,30 cm,groen,Noord,No
41,Arendshorst (old)_30,FF2 0-0193,418,NaN,NaN,"(, )",2023-07-04,2023-08-02 00:00:00,Boom,> 1 jaar,30 cm,groen,Noord,Yes
42,Lakenplein (old)_10,NETA1.0-0030,112,NaN,NaN,"(, )",2022-04-11,2023-08-02 00:00:00,Heesters,> 1 jaar,10 cm,groen,Binnenstad,Yes
43,Tasmanpark (2)_15,NETA1.0-0030,112,NaN,NaN,"(, )",2023-08-07,2025-01-01 00:00:00,Plantenvak,> 1 jaar,15 cm,groen,Binnenstad,Yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74040 entries, 0 to 74039
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   gateway_receive_time  74040 non-null  datetime64[ns]
 1   device                74040 non-null  int64         
 2   value                 74040 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.7 MB


...

,datum,windsnelheid,temperatuur,zonneschijn_duur,neerslag_duur,neerslag,luchtdruk,bewolking,vochtigheid,verdamping
27220,2025-07-11,31,200,101.0,0.0,0.0,10203,4.0,74.0,39.0
27221,2025-07-12,43,204,130.0,0.0,0.0,10153,6.0,71.0,50.0
27222,2025-07-13,25,203,26.0,6.0,59.0,10115,7.0,77.0,24.0
27223,2025-07-14,48,206,63.0,0.0,-1.0,10139,7.0,72.0,31.0
27224,2025-07-15,69,197,99.0,4.0,5.0,10160,6.0,59.0,41.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1199 entries, 0 to 1198
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   datum        1199 non-null   datetime64[ns]
 1   temperatuur  1199 non-null   float64       
 2   neerslag     1199 non-null   float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 28.2 KB


None

,datum,temperatuur,neerslag
0,2022-02-19,6.3,9.3
1,2022-02-20,8.1,22.2
2,2022-02-21,7.0,1.8
3,2022-02-22,7.8,1.5
4,2022-02-23,8.3,0.0


,datum,temperatuur,neerslag
1194,2025-07-07,17.1,0.0
1195,2025-07-08,15.6,14.7
1196,2025-07-09,19.0,0.0
1197,2025-07-10,19.0,0.0
1198,2025-07-11,19.0,0.0


1.3 Check latest incoming sensor data

In [11]:
latest_values = df_fact_sensor.merge(df_dim_sensor, how='left', left_on='device', right_on='device_id')
latest_values = latest_values.loc[latest_values.groupby('device')['gateway_receive_time'].idxmax()]
print(len(latest_values))

latest_values[['device_name','device','gateway_receive_time']].sort_values(by=['gateway_receive_time'], ascending=False)

26


,device_name,device,gateway_receive_time
134822,FF2 0-0148,372,2025-07-16 14:30:06
134821,FF2 0-0131,356,2025-07-16 14:27:48
134819,FF2 0-0191,416,2025-07-16 12:43:32
134578,FF2 0-0147,395,2025-07-16 11:55:41
134490,FF2 0-0136,361,2025-07-16 11:54:17
134550,FF2 0-0144,369,2025-07-16 10:52:18
134673,FF2 0-0190,415,2025-07-16 10:51:18
134775,FF2 0-0193,418,2025-07-16 10:01:22
134650,FF2 0-0174,397,2025-07-16 02:33:03
134666,FF2 0-0189,414,2025-07-16 01:44:15


### 2. Prepareer data
- 2.1 Filter oude locaties er uit
- 2.2 Koppel df_dim_sensor aan df_fact_sensor
- 2.3 Filter per locatie op begin datum plaatsing tot eind datum plaatsing. Locaties zonder einddatum krijgen datum in de toekomst, zodat alle data tot aan vandaag in de dataset zit.
- 2.4 Resample per locatie de tijdreekst op dagniveau. De sensoren geven meerdere sensorwaardes per dag (in principe elke 4 uur), maar dit gaat niet parallel over alle sensoren en voor elke sensor even consequent. Daarom wordt er per dag een gemiddelde, min en max waarde berekend. Daar gaan we de analyse mee doen. De tijdbox functie (add_6H_timebox) gebruiken we momenteel niet, omdat we op dagniveau gaan kijken. Mochten we goede weerdata en sensordata op (4-)uurniveau hebben, dan zouden we ook op een lagere datum_tijd granulariteit analyses kunnen doen.
- 2.5 Prepareer weerdata. De temperatuur en neerslag wordt van weerstation **Zusterhof** in Leiden gebruikt. De overige weerdata vanuit het KNMI van weerstation **Schiphol**. Wanneer de data uit Zusterhof NaN values bevat, wordt deze vervangen door data vanuit het KNMI. **Let op**: we vervangen de negatieve neerslagwaarden (-1) door 0. KNMI maakt onderscheid tussen 'geen regen' (= 0) en 'bijna geen regen' (= -1). In ons geval zien we 'bijna geen' regen als 'geen regen' 
- 2.6 Koppel weerdata aan sensordata
- 2.7 Toevoegen van berekende kolommen

In [12]:
# 2.1 Filter oude locaties er uit
df_dim_sensor = df_dim_sensor[df_dim_sensor['old'] == 'No']
df_dim_sensor.head()

,locatie,device_name,device_id,lat,lon,coordinaat,datum_geplaatst,datum_weggehaald,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old
7,Boshuizerkade_30,FF2 0-0131,356,NaN,NaN,"(, )",2023-07-04,2262-04-11 00:00:00,Plantenvak,0-1 jaar,30 cm,bebouwing,Zuid,No
10,Niet geplaatst,FF2 0-0135,360,NaN,NaN,"(, )",2025-05-22,2262-04-12 00:00:00,Plant en boom,> 1 jaar,15 cm,bebouwing,Midden,No
11,Tasmanpark (1)_15,FF2 0-0136,361,NaN,NaN,"(, )",2023-07-04,2262-04-11 00:00:00,Plant en boom,> 1 jaar,15 cm,groen,Binnenstad,No
15,Molen Lammermarkt_15,FF2 0-0144,369,NaN,NaN,"(, )",2023-07-04,2262-04-11 00:00:00,Plantenvak,> 1 jaar,15 cm,bebouwing,Binnenstad,No
16,Bachstraat (2)_15,FF2 0-0147,395,NaN,NaN,"(, )",2023-08-07,2262-04-11 00:00:00,Plantenvak,0-1 jaar,15 cm,bebouwing,Zuid,No


In [18]:
# 2.2 Koppel df_dim_sensor aan df_fact_sensor
df = df_dim_sensor.merge(df_fact_sensor, how='left', left_on=[
                         'device_id'], right_on=['device']).drop(columns='device')

In [19]:
# 2.3 Filter per locatie op begin datum plaatsing tot eind datum plaatsing.

# Filter time series tussen 'datum_geplaatst' en 'datum_opgehaald'
df = df[(df['gateway_receive_time'] > df['datum_geplaatst']) &
        (df['gateway_receive_time'] < df['datum_weggehaald'])]
df.drop(columns=['device_id', 'lat','lon','coordinaat','datum_geplaatst','datum_weggehaald'], inplace=True)

# # # Rename columns
# df.columns = ['locatie', 'device_name', 'soort_plant', 'leeftijd_plant', 'diepte_plaatsing', 'extra_omschrijving', 'locatie_regio',
#               'old', 'current_location', 'device_name_org', 'datum_tijd', 'meetwaarde']

df.columns = ['locatie', 'device_name', 'soort_plant', 'leeftijd_plant',
       'diepte_plaatsing', 'omgevingsfactoren', 'locatie_regio', 'old',
       'datum_tijd', 'meetwaarde']

# Verzamel locaties
locaties_lst = list(df['locatie'].unique())

# Inspecteer nieuwe dataset
display(df.info(5))
display(df.head(5))
display(df.tail(5))

<class 'pandas.core.frame.DataFrame'>
Index: 29656 entries, 0 to 37281
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   locatie            29656 non-null  object        
 1   device_name        29656 non-null  object        
 2   soort_plant        29656 non-null  object        
 3   leeftijd_plant     29656 non-null  object        
 4   diepte_plaatsing   29656 non-null  object        
 5   omgevingsfactoren  29656 non-null  object        
 6   locatie_regio      29656 non-null  object        
 7   old                29656 non-null  object        
 8   datum_tijd         29656 non-null  datetime64[ns]
 9   meetwaarde         29656 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(8)
memory usage: 2.5+ MB


None

,locatie,device_name,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old,datum_tijd,meetwaarde
0,Boshuizerkade_30,FF2 0-0131,Plantenvak,0-1 jaar,30 cm,bebouwing,Zuid,No,2024-10-30 07:38:22,11.51
1,Boshuizerkade_30,FF2 0-0131,Plantenvak,0-1 jaar,30 cm,bebouwing,Zuid,No,2024-10-30 03:42:38,11.57
2,Boshuizerkade_30,FF2 0-0131,Plantenvak,0-1 jaar,30 cm,bebouwing,Zuid,No,2024-10-29 23:46:53,11.82
3,Boshuizerkade_30,FF2 0-0131,Plantenvak,0-1 jaar,30 cm,bebouwing,Zuid,No,2024-10-29 19:51:06,11.70
4,Boshuizerkade_30,FF2 0-0131,Plantenvak,0-1 jaar,30 cm,bebouwing,Zuid,No,2024-10-29 15:55:17,11.70


,locatie,device_name,soort_plant,leeftijd_plant,diepte_plaatsing,omgevingsfactoren,locatie_regio,old,datum_tijd,meetwaarde
37277,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,2025-07-12 06:47:09,5.18
37278,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,2025-07-12 02:49:06,5.20
37279,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,2025-07-11 22:51:02,5.25
37280,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,2025-07-11 18:52:51,5.13
37281,Arendshorst_30,FF2 0-0193,Boom,> 1 jaar,30 cm,groen,Noord,No,2025-07-11 14:54:29,5.23


In [ ]:
# 2.4 Resample per locatie de tijdreekst op dagniveau.

# Resample time series op dagniveau
df_resample = df.set_index(['locatie', 'datum_tijd'])
resample_freq = 'D'  # '6H'

df = resample(df_resample, resample_freq)
df.columns = ['locatie', 'datum', 'min_meetwaarde',
              'max_meetwaarde', 'gem_meetwaarde']

df = df_dim_sensor[df_dim_sensor['locatie'].isin(locaties_lst)].merge(
    df, how='left', left_on=['locatie'], right_on=['locatie'])
df = df[(df['datum'] > df['datum_geplaatst']) &
        (df['datum'] < df['datum_weggehaald'])]

df = df[['locatie', 'diepte_plaatsing', 'leeftijd_plant', 'soort_plant', 'locatie_regio',
       'omgevingsfactoren', 'datum','min_meetwaarde', 'max_meetwaarde', 'gem_meetwaarde']]
       
df.head()

/Users/jspan/Library/CloudStorage/OneDrive-ilionxGroupBV/03 Projects/05 - Gemeente Leiden/leiden_water_in_the_cloud/src/logic_components/data_transformations.py:44: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



,locatie,diepte_plaatsing,leeftijd_plant,soort_plant,locatie_regio,omgevingsfactoren,datum,min_meetwaarde,max_meetwaarde,gem_meetwaarde
1,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-05,4.84,12.10,8.728333
2,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-06,10.32,11.39,10.823333
3,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-07,9.16,10.16,9.663333
4,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-08,8.47,9.16,8.856667
5,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-09,8.05,8.45,8.275000


In [21]:
# Step 1: Filter df_KNMI to include only data after January 1, 2022
df_weerdata = df_KNMI[df_KNMI['datum'] > '2022-01-01']

# Step 2: Replace NaN values in 'temperatuur' and 'neerslag' columns
df_weerdata['temperatuur'] = np.where(df_weerdata['temperatuur'].isna(), 0, df_weerdata['temperatuur'] / 10)
df_weerdata['neerslag'] = np.where(df_weerdata['neerslag'].isna(), 0, df_weerdata['neerslag'] / 10)

# Step 3: Replace -0.1 values in 'neerslag' column with 0
df_weerdata.loc[df_weerdata['neerslag'] < 0, 'neerslag'] = 0

# Display the modified df_weerdata to user (this would be for testing in an actual use case)
df_weerdata.head()

/var/folders/3_/n5vy3f4s457_v705xh5_zx9w0000gn/T/ipykernel_4576/1997770681.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/3_/n5vy3f4s457_v705xh5_zx9w0000gn/T/ipykernel_4576/1997770681.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,datum,windsnelheid,temperatuur,zonneschijn_duur,neerslag_duur,neerslag,luchtdruk,bewolking,vochtigheid,verdamping
25934,2022-01-02,90,11.4,20.0,51.0,6.5,10101,7.0,85.0,3.0
25935,2022-01-03,85,9.4,30.0,0.0,0.0,10066,5.0,84.0,4.0
25936,2022-01-04,45,6.4,5.0,9.0,0.7,9977,7.0,84.0,2.0
25937,2022-01-05,88,5.5,15.0,45.0,3.5,10062,6.0,80.0,2.0
25938,2022-01-06,43,3.3,46.0,10.0,0.2,10165,5.0,84.0,4.0


In [22]:
#2.6 koppel weerdata aan sensordata
df = df.merge(df_weerdata)
df.head(5)

,locatie,diepte_plaatsing,leeftijd_plant,soort_plant,locatie_regio,omgevingsfactoren,datum,min_meetwaarde,max_meetwaarde,gem_meetwaarde,windsnelheid,temperatuur,zonneschijn_duur,neerslag_duur,neerslag,luchtdruk,bewolking,vochtigheid,verdamping
0,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-05,4.84,12.10,8.728333,90,14.2,17.0,97.0,26.6,10074,8.0,87.0,14.0
1,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-06,10.32,11.39,10.823333,45,17.0,120.0,0.0,0.0,10171,7.0,70.0,43.0
2,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-07,9.16,10.16,9.663333,38,21.2,152.0,0.0,0.0,10189,2.0,56.0,55.0
3,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-08,8.47,9.16,8.856667,29,24.8,135.0,0.0,0.0,10164,3.0,58.0,54.0
4,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-09,8.05,8.45,8.275000,32,21.5,49.0,10.0,9.2,10188,8.0,78.0,32.0


In [23]:
df.rename(columns={'omgevingsfactoren': 'extra_omschrijving'}, inplace=True)
df.to_excel('./data/prepped_sensor_data.xlsx', index=False)
df.head(5)

,locatie,diepte_plaatsing,leeftijd_plant,soort_plant,locatie_regio,extra_omschrijving,datum,min_meetwaarde,max_meetwaarde,gem_meetwaarde,windsnelheid,temperatuur,zonneschijn_duur,neerslag_duur,neerslag,luchtdruk,bewolking,vochtigheid,verdamping
0,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-05,4.84,12.10,8.728333,90,14.2,17.0,97.0,26.6,10074,8.0,87.0,14.0
1,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-06,10.32,11.39,10.823333,45,17.0,120.0,0.0,0.0,10171,7.0,70.0,43.0
2,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-07,9.16,10.16,9.663333,38,21.2,152.0,0.0,0.0,10189,2.0,56.0,55.0
3,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-08,8.47,9.16,8.856667,29,24.8,135.0,0.0,0.0,10164,3.0,58.0,54.0
4,Boshuizerkade_30,30 cm,0-1 jaar,Plantenvak,Zuid,bebouwing,2023-07-09,8.05,8.45,8.275000,32,21.5,49.0,10.0,9.2,10188,8.0,78.0,32.0


In [24]:
df_KNMI.to_excel('./data/KNMI.xlsx', index=False)
df_KNMI.head(5)

,datum,windsnelheid,temperatuur,zonneschijn_duur,neerslag_duur,neerslag,luchtdruk,bewolking,vochtigheid,verdamping
0,1951-01-01,87,12,NaN,NaN,NaN,9891,7.0,90.0,NaN
1,1951-01-02,41,13,NaN,NaN,NaN,9876,8.0,93.0,NaN
2,1951-01-03,21,3,NaN,NaN,NaN,10019,6.0,94.0,NaN
3,1951-01-04,77,12,NaN,NaN,NaN,10098,7.0,94.0,NaN
4,1951-01-05,87,48,NaN,NaN,NaN,10059,8.0,95.0,NaN
